# DSFB-DSCD threshold sweep replay

Generate a run directory from the workspace root with:

```bash
cargo run --release -p dsfb-dscd -- --num-events 8192 --tau-steps 201
```

By default, the next cell tries several ways to locate DSCD outputs and auto-selects the newest run.
If none are found, it can auto-bootstrap by running `dsfb-dscd` when a workspace and `cargo` are available.
In Colab-like runtimes, it can also auto-clone the DSFB repo before bootstrapping.
Bootstrap logs now include a phase-based percentage indicator like `[ 62%] dsfb-dscd sweep running...`.
You can force paths with environment variables:
- `DSFB_DSCD_RUN_DIR`: absolute path to one specific run folder.
- `DSFB_DSCD_OUTPUT_ROOT`: absolute path to the `output-dsfb-dscd` folder.
- `DSFB_WORKSPACE_ROOT`: absolute path to the DSFB workspace root.
- `DSFB_DSCD_AUTO_INSTALL_RUST`: set to `0` to disable auto-installing Rust toolchain.



In [ ]:
from pathlib import Path
from typing import List, Optional, Tuple
import os
import shutil
import subprocess
import time


DEFAULT_REPO_URL = "https://github.com/infinityabundance/dsfb.git"


def _status(percent: int, message: str) -> None:
    pct = max(0, min(100, int(percent)))
    print(f"[{pct:3d}%] {message}")


def _progress_tick_seconds() -> float:
    raw = os.environ.get("DSFB_DSCD_PROGRESS_TICK_SECONDS", "5")
    try:
        value = float(raw)
    except ValueError:
        return 5.0
    return max(1.0, value)


def _run_with_progress(
    cmd: List[str],
    *,
    cwd: Optional[Path],
    start_pct: int,
    end_pct: int,
    label: str,
) -> None:
    start = max(0, min(100, int(start_pct)))
    end = max(start, min(100, int(end_pct)))

    _status(start, f"{label} started")
    proc = subprocess.Popen(cmd, cwd=str(cwd) if cwd is not None else None)

    current = start
    span = max(1, end - start)
    step = max(1, span // 8)
    tick_seconds = _progress_tick_seconds()

    while True:
        rc = proc.poll()
        if rc is not None:
            if rc != 0:
                raise subprocess.CalledProcessError(rc, cmd)
            break

        time.sleep(tick_seconds)
        if end > start:
            current = min(end - 1, current + step)
            _status(current, f"{label} running...")

    _status(end, f"{label} complete")


def _is_workspace_root(path: Path) -> bool:
    return (path / "Cargo.toml").exists() and (path / "crates" / "dsfb-dscd").exists()


def _discover_workspace_root() -> Optional[Path]:
    env_workspace = os.environ.get("DSFB_WORKSPACE_ROOT")
    if env_workspace:
        candidate = Path(env_workspace).expanduser().resolve()
        if _is_workspace_root(candidate):
            return candidate

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_workspace_root(candidate):
            return candidate

    search_roots = [
        Path.home(),
        Path("/home"),
        Path("/content"),
        Path("/workspace"),
        Path("/workspaces"),
        Path("/kaggle/working"),
    ]
    for root in search_roots:
        if not root.exists():
            continue

        if _is_workspace_root(root):
            return root

        for pattern in ("*/Cargo.toml", "*/*/Cargo.toml", "*/*/*/Cargo.toml", "*/*/*/*/Cargo.toml"):
            for cargo_toml in root.glob(pattern):
                candidate = cargo_toml.parent
                if _is_workspace_root(candidate):
                    return candidate

    return None


def _dedupe_paths(paths: List[Path]) -> List[Path]:
    out: List[Path] = []
    seen = set()
    for path in paths:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        out.append(path)
    return out


def _is_run_dir(path: Path) -> bool:
    return path.is_dir() and (path / "threshold_sweep.csv").exists()


def _candidate_output_roots(workspace_root: Optional[Path]) -> List[Path]:
    env_output_root = os.environ.get("DSFB_DSCD_OUTPUT_ROOT")

    roots: List[Path] = []

    if env_output_root:
        roots.append(Path(env_output_root).expanduser())

    if workspace_root is not None:
        roots.append(workspace_root / "output-dsfb-dscd")
        roots.append(workspace_root / "crates" / "dsfb-dscd" / "output-dsfb-dscd")

    cwd = Path.cwd().resolve()
    roots.append(cwd / "output-dsfb-dscd")
    roots.extend(parent / "output-dsfb-dscd" for parent in cwd.parents)

    roots.extend([
        Path.home() / "output-dsfb-dscd",
        Path("/content/output-dsfb-dscd"),
        Path("/content/dsfb/output-dsfb-dscd"),
        Path("/content/dsfb/crates/dsfb-dscd/output-dsfb-dscd"),
        Path("/workspace/output-dsfb-dscd"),
        Path("/workspaces/output-dsfb-dscd"),
        Path("/tmp/output-dsfb-dscd"),
    ])

    return _dedupe_paths([path.resolve() for path in roots])


def _select_latest_run(output_roots: List[Path]) -> Tuple[Optional[Path], Optional[Path]]:
    for root in output_roots:
        if _is_run_dir(root):
            return root, root.parent

        if not root.exists() or not root.is_dir():
            continue

        run_candidates = sorted(path for path in root.iterdir() if _is_run_dir(path))
        if run_candidates:
            return run_candidates[-1], root

    return None, None


def _ensure_cargo_bin() -> Optional[str]:
    cargo_bin = shutil.which("cargo")
    if cargo_bin:
        _status(45, f"Using cargo binary: {cargo_bin}")
        return cargo_bin

    cargo_home_bin = Path.home() / ".cargo" / "bin" / "cargo"
    if cargo_home_bin.exists():
        _status(45, f"Using cargo binary: {cargo_home_bin}")
        return str(cargo_home_bin)

    auto_install = os.environ.get("DSFB_DSCD_AUTO_INSTALL_RUST", "1").strip().lower()
    if auto_install in {"0", "false", "no"}:
        return None

    try:
        _status(30, "cargo not found; installing Rust toolchain with rustup")
        _run_with_progress(
            ["bash", "-lc", "curl https://sh.rustup.rs -sSf | sh -s -- -y --profile minimal"],
            cwd=None,
            start_pct=32,
            end_pct=50,
            label="Rust toolchain install",
        )
    except Exception as exc:
        print(f"Rust install failed: {exc}")
        return None

    cargo_bin = shutil.which("cargo")
    if cargo_bin:
        _status(50, f"Using cargo binary: {cargo_bin}")
        return cargo_bin
    if cargo_home_bin.exists():
        _status(50, f"Using cargo binary: {cargo_home_bin}")
        return str(cargo_home_bin)
    return None


def _ensure_workspace_for_bootstrap(workspace_root: Optional[Path]) -> Optional[Path]:
    if workspace_root is not None and _is_workspace_root(workspace_root):
        _status(20, f"Using workspace: {workspace_root}")
        return workspace_root

    auto_clone = os.environ.get("DSFB_DSCD_AUTO_CLONE_REPO", "1").strip().lower()
    if auto_clone in {"0", "false", "no"}:
        return workspace_root

    git_bin = shutil.which("git")
    if git_bin is None:
        return workspace_root

    clone_base = Path("/content") if Path("/content").exists() else Path.cwd().resolve()
    default_target = clone_base / "dsfb"
    target = Path(os.environ.get("DSFB_WORKSPACE_ROOT", str(default_target))).expanduser().resolve()

    if _is_workspace_root(target):
        _status(20, f"Using workspace: {target}")
        return target

    if target.exists() and any(target.iterdir()):
        print(f"Auto-clone skipped; target exists and is non-empty: {target}")
        return workspace_root

    try:
        target.parent.mkdir(parents=True, exist_ok=True)
        repo_url = os.environ.get("DSFB_REPO_URL", DEFAULT_REPO_URL)
        _status(10, f"Cloning DSFB repository for bootstrap: {repo_url} -> {target}")
        _run_with_progress(
            [git_bin, "clone", "--depth", "1", repo_url, str(target)],
            cwd=None,
            start_pct=12,
            end_pct=24,
            label="Repository clone",
        )
    except Exception as exc:
        print(f"Auto-clone failed: {exc}")
        return workspace_root

    if _is_workspace_root(target):
        _status(24, f"Using workspace: {target}")
        return target
    return workspace_root


def _auto_bootstrap_run(workspace_root: Optional[Path]) -> Tuple[bool, Optional[Path]]:
    disable = os.environ.get("DSFB_DSCD_DISABLE_AUTO_BOOTSTRAP", "").strip().lower()
    if disable in {"1", "true", "yes"}:
        return False, workspace_root

    _status(5, "No DSCD output found; starting bootstrap flow")

    workspace_root = _ensure_workspace_for_bootstrap(workspace_root)
    if workspace_root is None or not _is_workspace_root(workspace_root):
        _status(25, "No DSFB workspace available for bootstrap")
        return False, workspace_root

    cargo_bin = _ensure_cargo_bin()
    if cargo_bin is None:
        _status(55, "cargo unavailable; skipping auto-bootstrap run")
        return False, workspace_root

    num_events = os.environ.get("DSFB_DSCD_BOOTSTRAP_NUM_EVENTS", "8192")
    tau_steps = os.environ.get("DSFB_DSCD_BOOTSTRAP_TAU_STEPS", "201")
    cmd = [
        cargo_bin,
        "run",
        "--release",
        "-p",
        "dsfb-dscd",
        "--",
        "--num-events",
        str(num_events),
        "--tau-steps",
        str(tau_steps),
    ]

    _status(60, "Running dsfb-dscd sweep to generate output")
    print("+", " ".join(cmd), f"(cwd={workspace_root})")
    try:
        _run_with_progress(cmd, cwd=workspace_root, start_pct=62, end_pct=95, label="dsfb-dscd sweep")
    except Exception as exc:
        print(f"Auto-bootstrap failed: {exc}")
        return False, workspace_root

    _status(100, "Bootstrap run completed")
    return True, workspace_root


WORKSPACE_ROOT = _discover_workspace_root()
OUTPUT_ROOTS = _candidate_output_roots(WORKSPACE_ROOT)

RUN_DIR_ENV = os.environ.get("DSFB_DSCD_RUN_DIR")

if RUN_DIR_ENV is not None:
    RUN_DIR = Path(RUN_DIR_ENV).expanduser().resolve()
    if not _is_run_dir(RUN_DIR):
        raise FileNotFoundError(
            f"Configured DSFB_DSCD_RUN_DIR is not a valid DSCD run folder: {RUN_DIR}. "
            "Expected threshold_sweep.csv inside that directory."
        )
    OUTPUT_ROOT = RUN_DIR.parent
    _status(100, "Using DSFB_DSCD_RUN_DIR override")
else:
    RUN_DIR, OUTPUT_ROOT = _select_latest_run(OUTPUT_ROOTS)
    if RUN_DIR is None:
        bootstrapped, WORKSPACE_ROOT = _auto_bootstrap_run(WORKSPACE_ROOT)
        if bootstrapped:
            OUTPUT_ROOTS = _candidate_output_roots(WORKSPACE_ROOT)
            RUN_DIR, OUTPUT_ROOT = _select_latest_run(OUTPUT_ROOTS)
    else:
        _status(100, "Using latest existing DSCD run")

    if RUN_DIR is None:
        searched = "\n".join(f" - {path}" for path in OUTPUT_ROOTS)
        raise FileNotFoundError(
            "No DSCD output folders found. Set DSFB_DSCD_RUN_DIR to a specific run folder, "
            "or DSFB_DSCD_OUTPUT_ROOT to an output root. Searched:\n" + searched
        )

print(f"Workspace root: {WORKSPACE_ROOT}")
print(f"Output roots searched: {len(OUTPUT_ROOTS)}")
print(f"Using OUTPUT_ROOT: {OUTPUT_ROOT}")
print(f"Using RUN_DIR: {RUN_DIR}")
RUN_DIR



In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

threshold_df = pd.read_csv(RUN_DIR / 'threshold_sweep.csv')
threshold_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_df['tau'], threshold_df['expansion_ratio'], linewidth=2)
ax.set_title('DSCD expansion ratio vs trust threshold')
ax.set_xlabel('tau')
ax.set_ylabel('expansion_ratio')
ax.grid(alpha=0.25)
plt.show()

In [ ]:
finite_size_path = RUN_DIR / 'finite_size_scaling.csv'
if finite_size_path.exists():
    finite_df = pd.read_csv(finite_size_path)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(finite_df['num_events'], finite_df['transition_width'], marker='o')
    axes[0].set_title('Transition width vs N')
    axes[0].set_xlabel('num_events')
    axes[0].set_ylabel('transition_width')
    axes[0].grid(alpha=0.25)

    axes[1].plot(finite_df['num_events'], finite_df['max_derivative'], marker='o')
    axes[1].set_title('Max derivative vs N')
    axes[1].set_xlabel('num_events')
    axes[1].set_ylabel('max_derivative')
    axes[1].grid(alpha=0.25)
    plt.show()
else:
    print('finite_size_scaling.csv not present for this run')

In [ ]:
events_path = RUN_DIR / 'graph_events.csv'
edges_path = RUN_DIR / 'graph_edges.csv'

events_df = pd.read_csv(events_path)
edges_df = pd.read_csv(edges_path)

subset_edges = edges_df.head(128)
subset_nodes = sorted(set(subset_edges['from_event_id']).union(subset_edges['to_event_id']))

graph = nx.DiGraph()
graph.add_nodes_from(subset_nodes)
graph.add_edges_from(subset_edges[['from_event_id', 'to_event_id']].itertuples(index=False, name=None))

plt.figure(figsize=(10, 6))
pos = nx.spring_layout(graph, seed=7)
nx.draw_networkx(graph, pos=pos, node_size=120, with_labels=False, arrows=True)
plt.title('DSCD graph preview')
plt.axis('off')
plt.show()